# Silver → Gold Transformation

Reads cleansed Delta from the **Silver** layer, renames all columns from `PascalCase`
to `snake_case`, and writes the result as **Delta** to the Gold layer.

> **Requires:** Run `storagemount` first (or ensure OAuth2 Spark config is set on the cluster).

In [ ]:
%run ./autoload

In [ ]:
dbutils.widgets.text("storage_account", "sadataeng260524dev", "Storage Account")
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
auth  = bobydo.AdlsAuth(dbutils, spark)
paths = auth.setup(STORAGE_ACCOUNT)
BRONZE, SILVER, GOLD = paths["BRONZE"], paths["SILVER"], paths["GOLD"]

## 1. Verify Silver contents

In [ ]:
display(dbutils.fs.ls(f"{SILVER}/Sales/"))

## 2. Single-table preview (Address)

In [ ]:
df = spark.read.format('delta').load(f"{SILVER}/Sales/Address/")
display(df)

## 3. Column rename helper: PascalCase → snake_case

In [ ]:
def rename_columns_to_snake_case(df):
    """
    Convert column names from PascalCase or camelCase to snake_case in a PySpark DataFrame.

    Args:
        df (DataFrame): The input DataFrame with columns to be renamed.

    Returns:
        DataFrame: A new DataFrame with column names converted to snake_case.
    """
    rename_map = {}

    for old_col_name in df.columns:
        new_col_name = "".join([
            "_" + char.lower() if (
                char.isupper()
                and idx > 0
                and not old_col_name[idx - 1].isupper()
            ) else char.lower()
            for idx, char in enumerate(old_col_name)
        ]).lstrip("_")

        if new_col_name in rename_map.values():
            raise ValueError(f"Duplicate column name after renaming: '{new_col_name}'")

        rename_map[old_col_name] = new_col_name

    for old_col_name, new_col_name in rename_map.items():
        df = df.withColumnRenamed(old_col_name, new_col_name)

    return df


# Single-table test
df = rename_columns_to_snake_case(df)
display(df)

## 4. Transform all Sales tables → Gold

In [ ]:
# Discover all tables from silver
table_names = [f.name.split('/')[0] for f in dbutils.fs.ls(f"{SILVER}/Sales/")]
print("Tables found:", table_names)

for table in table_names:
    input_path  = f"{SILVER}/Sales/{table}/"
    output_path = f"{GOLD}/Sales/{table}/"

    df = spark.read.format('delta').load(input_path)
    df = rename_columns_to_snake_case(df)
    df.write.format('delta').mode('overwrite').save(output_path)
    print(f"  ✅ {table} → {output_path}")

print("\n✅ Silver → Gold complete")

## 5. Verify Gold output

In [ ]:
display(dbutils.fs.ls(f"{GOLD}/Sales/"))

In [ ]:
# Spot-check: preview last processed table
display(df)

## 6. Debug helper — schema mismatch check

If you see a schema error on re-runs, use this to compare silver vs gold schemas for a specific table.

In [ ]:
# Replace 'Address' with the table causing the issue
table = 'Address'

print("── Silver schema ──")
spark.read.format('delta').load(f"{SILVER}/Sales/{table}/").printSchema()

print("── Gold schema ──")
spark.read.format('delta').load(f"{GOLD}/Sales/{table}/").printSchema()